# Exploratory Data Analysis & Quality Audit
**Project:** Hourly Direction Prediction for SOLUSDT Using Gradient Boosting

**Institution:** IMPACT Lab, Department of Computer Science & Engineering, IIT Patna

**Course Guidelines Alignment:** Steps 3, 4, 5 (Data Collection, Splitting & Leakage Prevention, EDA)

---
### Objective
This notebook conducts an in-depth data audit and exploratory data analysis of SOLUSDT 15-minute market data from Binance. In accordance with strict leakage-prevention standards, **all analyses and distributions are computed exclusively on the training period** (the first 70% of chronological candles).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

from src.config import SYMBOL, INTERVAL, HORIZON, THRESH, DATA_DIR
from src.features.build import make_features, make_labels, FEATURES
from src.data.split import build_dataset, split_bounds

print('Configuration and libraries successfully loaded.')

## 1. Dataset Ingestion & Quality Audit
We inspect data integrity, date range, missing values, duplicate records, and sampling cadence.

In [ ]:
csv_path = DATA_DIR / f'{SYMBOL}_{INTERVAL}.csv'
df = pd.read_csv(csv_path, parse_dates=['time'])
print(f'Total Candles: {len(df):,}')
print(f'Time Span: {df["time"].min()} to {df["time"].max()}')
print(f'Missing Values: {df.isnull().sum().sum()}')
print(f'Duplicate Timestamps: {df["open_time"].duplicated().sum()}')
df.head(5)

## 2. Chronological Split & Embargo Enforcement
To prevent look-ahead bias and autocorrelation leakage in time series forecasting, we split chronologically: 70% train, 15% validation, and 15% test, with an embargo gap of `HORIZON = 4` candles between each partition.

In [ ]:
_, full_data = build_dataset(df)
bounds = split_bounds(len(full_data))
train_data = full_data.iloc[bounds['train'][0]:bounds['train'][1]].copy()

print(f'Total Usable Candles: {len(full_data):,}')
print(f'Training Candles (70%): {len(train_data):,} from {train_data["time"].iloc[0]} to {train_data["time"].iloc[-1]}')
print(f'Validation Candles (15%): {bounds["val"][1] - bounds["val"][0]:,}')
print(f'Held-Out Test Candles (15%): {bounds["test"][1] - bounds["test"][0]:,}')

## 3. Target Distribution & Dead-Zone Mechanism
Moves within ±0.20% (|return| <= 0.002) are designated as **Neutral** noise candles. Clear directional moves (Up vs Down) represent 73.8% of training candles.

In [ ]:
counts = train_data['y'].value_counts().sort_index()
labels = ['Down (0)', 'Neutral (1)', 'Up (2)']
shares = counts / len(train_data) * 100

clear = train_data[train_data['y'] != 1]
up_share = (clear['y'] == 2).mean()

print(f'Down candles: {counts.get(0, 0):,} ({shares.get(0, 0):.1f}%)')
print(f'Neutral candles: {counts.get(1, 0):,} ({shares.get(1, 0):.1f}%)')
print(f'Up candles: {counts.get(2, 0):,} ({shares.get(2, 0):.1f}%)')
print(f'Clear move Up share: {up_share:.3f} (True Directional Baseline)')

## 4. Forward Return Properties & Heavy Tails
We examine the empirical distribution of 1-hour ahead returns across training candles.

In [ ]:
fwd = train_data['fwd'] * 100
print(f'Mean Forward 1h Return: {fwd.mean():+.4f}%')
print(f'Standard Deviation: {fwd.std():.4f}%')
print(f'Skewness: {fwd.skew():.2f}')
print(f'Excess Kurtosis: {fwd.kurtosis():.2f} (Heavy-tailed distribution)')

## 5. Feature Correlation Matrix & Multicollinearity
Examining the correlation among the 18 causal technical indicators to understand collinear clusters.

In [ ]:
corr = train_data[FEATURES].corr()
top_pairs = []
for i in range(len(FEATURES)):
    for j in range(i + 1, len(FEATURES)):
        top_pairs.append((FEATURES[i], FEATURES[j], corr.iloc[i, j]))
top_pairs.sort(key=lambda x: abs(x[2]), reverse=True)

print('Top 5 Most Correlated Feature Pairs:')
for f1, f2, c in top_pairs[:5]:
    print(f'  {f1:<12} and {f2:<12} : r = {c:+.3f}')

## 6. Conclusions & Protocol Verification
- **Credible Baseline:** 50.6% Up among clear moves confirms that a trivial 50% majority predictor is the appropriate reference baseline.
- **Non-Linear Model Selection:** Extreme leptokurtosis (kurtosis ~16.8) and collinear clusters support tree-based ensemble methods (LightGBM) over unconstrained linear regressors.
- **Zero Leakage:** No test-set data was accessed during this analysis.